# MINOR PROJECT

***
## SENTIMENT ANALYSIS
***

### STEPS:
        -- Load data
        -- text pre-processing
        -- tensors , datasets, dataloaders
        -- build rnn
        -- train and eval
***

In [55]:
import pandas as pd

In [56]:
df = pd.read_csv("IMDB Dataset.csv")

In [57]:
df.shape

(50000, 2)

In [58]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [59]:
df.drop_duplicates(inplace=True)

In [60]:
df.shape

(49582, 2)

### Text-preprocessing:
                -- convert to lowercase
                -- remove urls(http,https)
                -- remove html
                -- remove puntuation(.,?#)
                -- remove stopwords
                -- stemming - reduce words to base form
                -- encode sentiment
                -- vectorisation(TF-IDF)
***

In [61]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [62]:
# CONVERT TO LOWERCASE

df["review"] = df["review"].str.lower()

In [63]:
# REGEX (https://docs.python.org/3/library/re.html)
import re
## re.sub("pattern", "replacement", "string")

sample_txt = "abc is the abc word"
new_txt = re.sub("abc", "xyz", sample_txt)
print(new_txt)


# \S => non-white space characters
# \s => white space characters


# remove everything in the text [except things in this bracket]

# . => any character
# * => 0 or more characters

xyz is the xyz word


In [64]:
# REMOVING THE URLs

def remove_url(text):
    new = re.sub(r"http\S+", "",text)
    return new

df["review"] =df["review"].apply(remove_url)

In [65]:
def remove_html(text):
    new = re.sub(r"<.*?>", "", text) # remove any character inside thses <>, ? means greedy removal (remove only until >)
    return new

df["review"] = df["review"].apply(remove_html)

In [66]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [67]:
# REMOVING PUNCTUATIONS

In [68]:

def remove_punctuations(text):
    new = re.sub(r"[^A-Za-z0-9\s]", "", text) # ^ - means apart from these remove all other
    return new

df["review"] = df["review"].apply(remove_punctuations)

In [69]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [70]:
## REMOVING STOPWORDS

import nltk # natural language toolkit

nltk.download("punkt") # tokenisation
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\prash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\prash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [71]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [72]:
# example

sample_text = "i will analyze ur sentiments"
tokens = word_tokenize(sample_text)

In [73]:
tokens

['i', 'will', 'analyze', 'ur', 'sentiments']

In [74]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text 

In [75]:
df["review"] = df["review"].apply(remove_stopwords)

In [76]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti filming technique un...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [80]:
## STEMMING

# running => run

from nltk.stem import PorterStemmer # lib for stemming


def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [81]:
df.head()


,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti film techniqu unssum oldtim...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [83]:
# ENCODING 

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [84]:
y = df["sentiment"]

In [85]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### vectorization TF-IDF

In [86]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

In [87]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4018521 stored elements and shape (49582, 5000)>

In [88]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4018521 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3543)	0.05525624059671651
  (0, 2869)	0.09363569785687688
  (0, 4942)	0.1143462865526561
  (0, 3002)	0.4721257539665051
  (0, 1283)	0.13562935370662169
  (0, 2289)	0.049645630659897244
  (0, 1930)	0.07909092864896988
  (0, 3556)	0.09666152065838991
  (0, 1369)	0.061648686056296906
  (0, 1960)	0.06145605263397629
  (0, 1623)	0.07456802660334816
  (0, 4373)	0.041879745251798425
  (0, 4176)	0.1780422408421322
  (0, 3699)	0.03346830311117425
  (0, 4739)	0.26848378236692794
  (0, 3807)	0.044306447189662514
  (0, 4770)	0.11749890918420128
  (0, 1739)	0.037543774684505316
  (0, 4500)	0.0763407674477712
  (0, 3862)	0.17507374076823898
  (0, 1632)	0.06145605263397629
  (0, 1860)	0.07411515187798978
  (0, 3334)	0.06409411735798108
  (0, 3337)	0.08445670622566916
  (0, 3437)	0.07868305860931851
  :	:
  (49581, 4142)	0.07151774999031583
  (49581, 4893)	0.089889401802

***
## Dataset and DataLoader
***

In [90]:
from sklearn.model_selection import train_test_split

X_train, X_test , y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [91]:
X_train.shape

(39665, 5000)

In [92]:
X_test.shape

(9917, 5000)

In [97]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [94]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3217381 stored elements and shape (39665, 5000)>

In [95]:
# convert to numpy array

X_train = X_train.toarray()
X_test = X_test.toarray()

In [98]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [99]:
train_loader = DataLoader(train_set , shuffle=True, batch_size = 64)
test_loader = DataLoader(test_set , shuffle=True, batch_size = 64)

***
## BUILD RNN
***

In [103]:
import torch.nn as nn
import torch.optim as optim

In [112]:
class RNN(nn.Module):

    def __init__(self,input_size, hidden_size=128, num_layers = 1): # input_size =(39665) text value size
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        # to avoid reshaping complex operations we do batch_size = True
        # changing input format for pytorch from (seq_length, batch_size, inp_size) to (batch_size, seq_len, inp_size)

        # fc layer
        self.fc = nn.Linear(hidden_size,1) #linear combo(neurons , output)
        # its a many to one architecture

    def forward(self,x):
        
        # optional step (initializing hidden state)
        # shape(num_of_layer , batch size , hidden layer size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size) # x.size(0) = batch_size = 64

        out,_ = self.rnn(x,h0)
        #rnn return 2 values, we want the 1st value
        # 1st value = hidden state of all timesteps =>> shape of returned value(batch_size, seq_len , hidden_size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:,-1,:]) # since we dont need all the hidden state but the last time step
        # all batch data but only last timesteps

        return out

In [113]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss() # binary cross entropy loss since two outputs
optimizer = optim.Adam(model.parameters())

## Training the model

In [114]:
# squeeze => reduce one dimension
# unsqueeze =>> inc by one dimension

In [115]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction hence now in 3d
        outputs = model(Xb) # model expects 3d but tf-idf vectors are matrices and in 2 dimensions
        # output dimension (batch_size,1)
        
        #convert outputs to probabilities
        
        # since sigmoid expects 1d input, we squeeze
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,)


        loss = criterion(outputs , yb) # loss compute
        loss.backward() # back propagation
        optimizer.step() # weight update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")
        

epoch = 1/10 and loss = 0.3160153031349182
epoch = 2/10 and loss = 0.27588656544685364
epoch = 3/10 and loss = 0.2633795738220215
epoch = 4/10 and loss = 0.2642410695552826
epoch = 5/10 and loss = 0.2184094786643982
epoch = 6/10 and loss = 0.14271605014801025
epoch = 7/10 and loss = 0.23003706336021423
epoch = 8/10 and loss = 0.23252221941947937
epoch = 9/10 and loss = 0.29705721139907837
epoch = 10/10 and loss = 0.3551611602306366


In [118]:
# evaluate

model.eval()

with torch.no_grad():

    correct_vals =0
    tot_vals = 0

    for Xb,yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals+=yb.size(0)
        correct_vals +=(predicted ==yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.72148835333266
